# Витрины Greenplum — 30 заданий

Модуль строит внутренние аналитические таблицы и итоговую витрину крупнейших BUY/SELL-сделок. Источник контролируемый, а затем подход переносится на MOEX.

## Результаты обучения

После **Факты, измерения и витрины** вы должны объяснять физическое выполнение на coordinator/segments, связывать logical SQL с Motion/I/O/skew, выбирать дизайн по workload и доказывать решение измерениями.

## Ментальная модель

Grain факта определяет ключи и аддитивность. В MPP collocation факта с крупным JOIN важнее случайного «равномерного» ключа; малые dimensions можно реплицировать.

```text
client → coordinator (parse/optimize)
              │ dispatch slices
       ┌──────┼──────┐
       ▼      ▼      ▼
    segment segment segment
       └── Motion/interconnect ──┘
              │
              ▼
          coordinator
```
Coordinator не должен становиться местом обработки всех строк. Хороший план оставляет
scan/aggregate на сегментах и перемещает только необходимое.

## Данные и grain

MOEX trade grain, ticker/day/deal type. Полные схемы находятся в `data-catalog`. Общие external/raw объекты читаются, учебные результаты создаются только в `m_razhin`.

## Инженерный алгоритм

1. Назовите grain и ключ. 2. Оцените объём/cardinality. 3. Выберите distribution/storage/partition. 4. Предскажите Motion и I/O. 5. Создайте минимальный объект. 6. ANALYZE. 7. Снимите EXPLAIN и сегментные метрики. 8. Сверьте результат.

Сформулируйте grain сделки, детерминированно ранжируйте BUY/SELL, проектируйте dimensions и сверяйте витрину с raw по ключам и VALUE.

## Типичные ошибки

- Переносить правила PostgreSQL без учёта MPP.
- Выбирать distribution key только по высокой cardinality.
- Путать partitioning с distribution.
- Считать Broadcast всегда плохим, а Redistribute всегда допустимым.
- Сравнивать время единственного запуска без rows/Motion/I/O.
- Создавать external object с путём, доступным Windows, но не сегментам.

## Вопросы для самопроверки

1. Где физически лежит строка? 2. Какие slices выполнят сегменты? 3. Что и сколько передаёт Motion? 4. Как проявится skew? 5. Что произойдёт при повторной загрузке? 6. Как доказать результат из независимого источника?

In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 150
%sql postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex

## 1. Начинайте с grain

Гранулярность отвечает, чему соответствует одна строка. Она определяет ключ, допустимые measures и JOIN. «Дневная витрина» недостаточно: нужно сказать `одна строка на trade_date, secid, deal_type`.

## 2. Факт и измерение

Факт содержит события/измеримые показатели и внешние ключи. Измерение описывает контекст. Денормализация может ускорить serving, но источник атрибутов и история должны быть определены.

## 3. Звезда

Star schema соединяет крупный факт с компактными измерениями. В Greenplum физическая модель важна: факт распределяют по частому крупному JOIN/фильтру, маленькие dimensions могут быть replicated.

## 4. Business и surrogate key

SECID — business key, который приходит из источника. Surrogate key идентифицирует конкретную dimension row/version. Он нужен SCD2 и изоляции warehouse от изменения source identifiers.

## 5. Additive measures

VALUE и QUANTITY обычно суммируются по измерениям. PRICE не является additive: дневная цена требует open/high/low/close или weighted average. Нельзя складывать средние и проценты как обычные суммы.

## 6. Semi-additive

Balance/остаток можно суммировать по инструментам, но не по времени. Для временной оси выбирают last snapshot. Всегда описывайте агрегируемость каждой measure.

## 7. Top-N

Крупнейшая сделка выбирается оконным rank внутри `(date,secid,deal_type)`. `row_number` даёт одну строку, но нужен tie-breaker. `rank` сохраняет все ties и меняет grain/число строк.

## 8. Open и close

MIN/MAX(price) не дают open/close. Нужна первая/последняя сделка по deal_time с детерминированным order. High/low — max/min цены. Weighted price рассчитывается по agreed weight.

## 9. SCD2

Изменение атрибута закрывает текущую version и открывает новую. Fact lookup выполняется по business key и event time. JOIN только current dimension переписывает историю.

## 10. Distribution витрины

Serving запросы по одному SECID и диапазону дат могут выиграть от distribution by secid. Но GROUP BY day across all securities потребует Motion. Policy выбирают по самым дорогим регулярным потребителям.

## 11. Partitioning витрины

Дневной/месячный факт часто partitioned по trade date/month ради pruning и инкрементной замены. Не создавайте partition на каждый SECID. Distribution и partitioning решают разные оси.

## 12. AO column

Широкие serving marts с выбором части measures подходят AO column + compression. Малые dimensions и audit/control — heap или AO row по характеру изменений.

## 13. Материализация

View хранит запрос, table хранит результат. Сложный расчёт на каждом consumer query может быть дорог. Материализованная mart требует pipeline, freshness, reconciliation и повторяемого refresh.

## 14. Дневные и месячные зависимости

Месячная mart зависит от дневной или детального факта. Late trade меняет день, затем месяц и возможно сезонность. Pipeline должен вычислять downstream affected slices.

## 15. Quality mart

Quality — самостоятельный продукт: snapshot count, active count, distinct instruments, zero/rejected, freshness. Он помогает потребителю понять покрытие и ETL — остановить публикацию.

## 16. Serving contract

Стабильный VIEW может скрывать физическую таблицу/version. Consumer получает фиксированные имена/типы, а команда может пересобрать storage и атомарно переключить underlying object.

## 17. Publish

Статусы BUILDING→READY→PUBLISHED или FAILED отделяют расчёт от доступности. Публикация происходит только после quality/reconciliation. Audit хранит version, period, counts и timestamps.

## 18. Оптимизация

Проверяйте типовой запрос: ticker/day/type, pruning, Motion, выбранные колонки и runtime skew. Хорошая витрина оптимизирует потребление, а не только процесс загрузки.

## 19. Reconciliation

Для агрегата сумма VALUE/QUANTITY должна согласовываться с фактом в той же области. Top-1 намеренно не сохраняет общую сумму — для него проверяют число групп, принадлежность строк и max rule.

## 20. Порядок

Contract grain→dimensions→fact→quality→daily→monthly→serving→plan→incremental refresh→reconciliation→publish audit.

### Задание 1. `m_razhin.gpm_01_grain`

**Что сделать:** Создайте VIEW-контракт гранулярности витрины крупных сделок.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Одна строка должна быть однозначно описана.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_01_grain.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',1);

### Задание 2. `m_razhin.gpm_02_dim_board`

**Что сделать:** Создайте измерение режима торгов.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Малое измерение можно replicated.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_02_dim_board.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',2);

### Задание 3. `m_razhin.gpm_03_dim_security`

**Что сделать:** Создайте измерение инструмента с surrogate key.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

SECID остаётся business key.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_03_dim_security.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',3);

### Задание 4. `m_razhin.gpm_04_dim_date`

**Что сделать:** Создайте календарь диапазона source.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Добавьте месяц, квартал, weekday.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_04_dim_date.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',4);

### Задание 5. `m_razhin.gpm_05_fact_trade`

**Что сделать:** Создайте детальный факт сделок.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Distribution и partitioning выбираются по запросам.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_05_fact_trade.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',5);

### Задание 6. `m_razhin.gpm_06_fact_quality`

**Что сделать:** Проверьте ключи, NULL, положительные суммы и тип сделки.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Quality до публикации.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_06_fact_quality.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',6);

### Задание 7. `m_razhin.gpm_07_fact_join`

**Что сделать:** Свяжите факт с измерениями без потерь.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Контролируйте many-to-many.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_07_fact_join.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',7);

### Задание 8. `m_razhin.gpm_08_daily_base`

**Что сделать:** Создайте дневной агрегат по SECID/deal_type.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Определите measures и grain.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_08_daily_base.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',8);

### Задание 9. `m_razhin.gpm_09_top_trade`

**Что сделать:** Найдите крупнейшую сделку каждого SECID/type/day.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

row_number с tie-breaker.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_09_top_trade.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',9);

### Задание 10. `m_razhin.gpm_10_top_ties`

**Что сделать:** Сохраните число ничьих максимальной VALUE.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Бизнес должен решить ties.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_10_top_ties.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',10);

## Уровень 2 — MOEX marts

### Задание 11. `m_razhin.gpm_11_required_mart`

**Что сделать:** Создайте витрину полей исходного Greenplum-задания.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

BOARDID…TRADE_SESSION_DATE.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_11_required_mart.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',11);

### Задание 12. `m_razhin.gpm_12_mart_policy`

**Что сделать:** Создайте VIEW обоснования storage/distribution/partition.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Рекомендация измерима.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_12_mart_policy.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',12);

### Задание 13. `m_razhin.gpm_13_liquidity_daily`

**Что сделать:** Создайте дневную ликвидность: trades, quantity, value.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Одна строка SECID/day.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_13_liquidity_daily.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',13);

### Задание 14. `m_razhin.gpm_14_liquidity_rank`

**Что сделать:** Добавьте ранг инструмента по обороту внутри дня.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

dense_rank.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_14_liquidity_rank.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',14);

### Задание 15. `m_razhin.gpm_15_price_daily`

**Что сделать:** Создайте open/high/low/close по времени сделок.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Open/close требуют порядка.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_15_price_daily.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',15);

### Задание 16. `m_razhin.gpm_16_prev_close`

**Что сделать:** Добавьте close предыдущего активного дня.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

lag после дневной агрегации.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_16_prev_close.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',16);

### Задание 17. `m_razhin.gpm_17_price_change`

**Что сделать:** Рассчитайте absolute и percent change.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

NULLIF prev_close.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_17_price_change.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',17);

### Задание 18. `m_razhin.gpm_18_monthly`

**Что сделать:** Создайте месячную витрину ликвидности/цен.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Не усредняйте проценты без смысла.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_18_monthly.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',18);

### Задание 19. `m_razhin.gpm_19_seasonality`

**Что сделать:** Создайте сезонность по SECID/month_num.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Сравнивайте активные дни.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_19_seasonality.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',19);

### Задание 20. `m_razhin.gpm_20_data_quality`

**Что сделать:** Создайте дневную quality mart.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Snapshot,active,distinct,zero/rejected.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_20_data_quality.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',20);

## Уровень 3 — history, incremental refresh и serving

### Задание 21. `m_razhin.gpm_21_scd_security`

**Что сделать:** Создайте SCD2 инструмента.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Одна current version.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_21_scd_security.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',21);

### Задание 22. `m_razhin.gpm_22_asof_join`

**Что сделать:** Свяжите сделки с исторической версией security.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Event time внутри valid interval.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_22_asof_join.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',22);

### Задание 23. `m_razhin.gpm_23_late_trade`

**Что сделать:** Покажите перерасчёт дня при late trade.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Delete+insert slice.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_23_late_trade.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',23);

### Задание 24. `m_razhin.gpm_24_increment_daily`

**Что сделать:** Реализуйте инкремент дневной витрины.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Публикуйте только проверенный день.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_24_increment_daily.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',24);

### Задание 25. `m_razhin.gpm_25_increment_monthly`

**Что сделать:** Определите затронутый месяц и пересчитайте его.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Late day меняет month.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_25_increment_monthly.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',25);

### Задание 26. `m_razhin.gpm_26_reconciliation`

**Что сделать:** Сверьте факт и витрины по count/value.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Measures должны сохраняться согласно grain.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_26_reconciliation.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',26);

### Задание 27. `m_razhin.gpm_27_query_plan`

**Что сделать:** Проверьте pruning/Motion типового запроса витрины.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Фильтр ticker/day/type.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_27_query_plan.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',27);

### Задание 28. `m_razhin.gpm_28_serving_view`

**Что сделать:** Создайте стабильный serving VIEW над физической mart.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Consumer contract отделён от storage.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_28_serving_view.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',28);

### Задание 29. `m_razhin.gpm_29_publish_audit`

**Что сделать:** Создайте audit публикации версий витрины.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

READY→PUBLISHED/FAILED.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_29_publish_audit.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',29);

### Задание 30. `m_razhin.gpm_30_showcase`

**Что сделать:** Соберите итоговый каталог витрин с grain,SLA,policy,status.

Перед SQL запишите grain, key, measures и physical policy. После — проверьте uniqueness, reconciliation и типовой план.

<details><summary>Подсказка</summary>

Документируйте все созданные marts.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpm_30_showcase.

In [ ]:
%%sql
-- Ручная проверка grain/quality/reconciliation.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('data_marts',30);

## Прогресс

In [ ]:
%%sql
SELECT * FROM greenplum_training.progress WHERE module_name='data_marts' ORDER BY task_no;